# Alpaca Paper – 5‑Minute Guardian

This notebook monitors open positions and flattens them when **hard stop**, **trailing stop**, or **take‑profit** thresholds are hit. It loads expectations from your latest **infer** CSV, tracks entries locally in a **ledger CSV**, reconciles with the broker each pass, and prints a concise heartbeat every 5 minutes.

**Fill your paper keys in Cell 0** (or export them as environment variables before starting the kernel).

In [ ]:
# === Cell 0 — PAPER KEYS (set here for paper; move to env for live) ===
APCA_KEY_ID     = "PKUNRTQLNIJ4FWITDRXUCGZTPT"  # <-- your PAPER Key ID (or leave blank to use env)
APCA_SECRET_KEY = "6fGdcfwCzp8vHUYXacaoVLpPsknY5n4d9ngukbqFdYpU" # <-- put your PAPER Secret Key here (or leave blank to use env)


In [ ]:
# --- Session mode ---
# 'RTH' = regular hours only
# 'EXT' = pre/post + RTH (US equities: 4:00–20:00 ET)
# 'ALWAYS' = no time gating (useful for testing)
SESSION_MODE = 'ALWAYS'   # choose: 'RTH' | 'EXT' | 'ALWAYS'

from datetime import time as dtime

# PT times (your notebook is on America/Los_Angeles)
PRE_START_PT   = dtime(1,  0)   # 04:00 ET
RTH_START_PT   = dtime(6, 30)   # 09:30 ET
RTH_END_PT     = dtime(13, 0)   # 16:00 ET
POST_END_PT    = dtime(17, 0)   # 20:00 ET

def within_session(now_local) -> bool:
    if SESSION_MODE == 'ALWAYS':
        return True
    t = now_local.timetz().replace(tzinfo=None)
    in_rth  = RTH_START_PT <= t <= RTH_END_PT
    if SESSION_MODE == 'RTH':
        return in_rth
    in_pre  = PRE_START_PT <= t < RTH_START_PT
    in_post = RTH_END_PT   <  t <= POST_END_PT
    return in_pre or in_rth or in_post


In [ ]:
# Preview thresholds for sanity
STOP_PCT  = globals().get("HARD_STOP_PCT", 0.025) #stop loss at 2.5%
TRAIL_PCT = globals().get("TRAIL_FROM_PEAK_PCT", 0.03) # 3% drop from peak 
TP_PCT    = globals().get("TP_PCT", 0.013) # 13% lock in gains

# Global risk knobs (used if a row has no overrides) aligning to so %s match
HARD_STOP_PCT  = STOP_PCT     
TRAIL_FROM_PEAK_PCT  = TRAIL_PCT
TP_PCT    = TP_PCT

In [ ]:
# === Cell 1 — Alpaca config & WhoAmI ===
import os, requests, pandas as pd, numpy as np, asyncio, time, glob, uuid
from pathlib import Path
from datetime import datetime, time as dtime, timedelta
from dateutil import tz

TRADING_BASE = 'https://paper-api.alpaca.markets'
DATA_BASE    = 'https://data.alpaca.markets'

# Prefer in-notebook keys; fall back to environment variables
if 'APCA_KEY_ID' in globals() and APCA_KEY_ID:
    APCA_HEADERS = {
        'APCA-API-KEY-ID': APCA_KEY_ID,
        'APCA-API-SECRET-KEY': APCA_SECRET_KEY,
        'Content-Type': 'application/json',
    }
else:
    APCA_HEADERS = {
        'APCA-API-KEY-ID': os.getenv('APCA_API_KEY_ID', ''),
        'APCA-API-SECRET-KEY': os.getenv('APCA_API_SECRET_KEY', ''),
        'Content-Type': 'application/json',
    }

assert APCA_HEADERS['APCA-API-KEY-ID'] and APCA_HEADERS['APCA-API-SECRET-KEY'], (
    'Set APCA_KEY_ID/APCA_SECRET_KEY in Cell 0 or export APCA_API_KEY_ID/APCA_API_SECRET_KEY env vars.'
)

# Timezone (Pacific)
TZ = tz.gettz('America/Los_Angeles')
MARKET_START_PT = dtime(6, 35)   # 9:35 ET (skip opening minute)
MARKET_END_PT   = dtime(13, 0)   # 16:00 ET

# (Optional) replace the WhoAmI block with this slightly richer version
try:
    r = requests.get(f"{TRADING_BASE}/v2/account", headers=APCA_HEADERS, timeout=10)
    r.raise_for_status()
    acct = r.json()
    acct_id   = acct.get('id')
    status    = acct.get('status')
    equity    = acct.get('equity')
    blocked   = acct.get('trading_blocked')
    pattern   = acct.get('pattern_day_trader')  # FYI flag; not critical for paper
    multiplier= acct.get('multiplier')          # margin multiplier (often "2" on paper margin, "1" cash)

    print(f"Account ID: {acct_id} | Status: {status} | Equity: {equity}")
    if blocked:
        print("⚠️  trading_blocked = True (Alpaca will reject orders).")
    if status != 'ACTIVE':
        print("⚠️  Account status is not ACTIVE; orders may be rejected.")

except requests.HTTPError as e:
    print("❌ Account check failed:", getattr(e.response, "text", str(e)))
    raise


In [ ]:
# === Cell 2 — Build/refresh LEDGER & WATCHLIST from Alpaca positions ===
import requests, pandas as pd
from pathlib import Path
from datetime import datetime

LEDGER_PATH = Path("./alpaca_paper/ledger.csv")
GUARD_OUT   = Path("./alpaca_paper/out"); GUARD_OUT.mkdir(parents=True, exist_ok=True)

# If you already defined last_price() earlier, we’ll use it; otherwise a tiny fallback:
def _last_price_fallback(sym: str) -> float | None:
    try:
        r = requests.get(f"{DATA_BASE}/v2/stocks/{sym}/trades/latest", headers=APCA_HEADERS, timeout=5)
        if r.status_code == 200:
            return float(r.json()["trade"]["p"])
    except Exception:
        pass
    try:
        r = requests.get(f"{DATA_BASE}/v2/stocks/{sym}/bars/latest", headers=APCA_HEADERS, timeout=5)
        if r.status_code == 200:
            return float(r.json()["bar"]["c"])
    except Exception:
        pass
    return None

_last_px = globals().get("last_price", _last_price_fallback)

def _positions_df() -> pd.DataFrame:
    r = requests.get(f"{TRADING_BASE}/v2/positions", headers=APCA_HEADERS, timeout=10)
    if r.status_code == 404:
        return pd.DataFrame(columns=["symbol","qty","avg_entry_price"])
    r.raise_for_status()
    js = r.json()
    if not js:
        return pd.DataFrame(columns=["symbol","qty","avg_entry_price"])
    df = pd.DataFrame(js)
    keep = [c for c in ("symbol","qty","avg_entry_price","asset_id","fractionable","asset_class")]
    keep = [c for c in keep if c in df.columns]
    return df[keep].copy()

def build_ledger_from_broker() -> pd.DataFrame:
    global LEDGER
    p = _positions_df()
    if p.empty:
        LEDGER = pd.DataFrame(columns=["symbol","shares","entry_px","peak_px","last_update"])
        LEDGER.to_csv(LEDGER_PATH, index=False)
        print("No open positions at broker; LEDGER is empty.")
        return LEDGER

    # Normalize/compute
    p["symbol"]  = p["symbol"].astype(str).str.upper()
    p["shares"]  = p["qty"].astype(float)
    p["entry_px"] = p["avg_entry_price"].astype(float)

    # Load existing peaks if present
    if LEDGER_PATH.exists():
        old = pd.read_csv(LEDGER_PATH)
        if not old.empty and "symbol" in old.columns:
            old["symbol"] = old["symbol"].astype(str).str.upper()
            p = p.merge(old[["symbol","peak_px"]], on="symbol", how="left")

    # Initialize peak for new names: max(entry, current)
    peaks = []
    for _, r in p.iterrows():
        px_now = _last_px(r["symbol"])
        if pd.notna(r.get("peak_px")):
            peaks.append(float(r["peak_px"]))
        else:
            base = r["entry_px"]
            peaks.append(max(base, px_now) if px_now else base)
    p["peak_px"] = peaks
    p["last_update"] = datetime.now(TZ).isoformat(timespec="seconds")

    LEDGER = p[["symbol","shares","entry_px","peak_px","last_update"]].copy()
    LEDGER.to_csv(LEDGER_PATH, index=False)

    # WATCHLIST is simply the held symbols
    wl = sorted(LEDGER["symbol"].tolist())
    globals()["WATCHLIST"] = wl


    
    prev = []
    for _, r in LEDGER.head(12).iterrows():
        prev.append({
            "symbol": r["symbol"],
            "entry": r["entry_px"],
            "peak":  r["peak_px"],
            "hard":  r["entry_px"] * (1 - STOP_PCT),
            "trail": r["peak_px"]  * (1 - TRAIL_PCT),
            "tp":    r["entry_px"] * (1 + TP_PCT),
        })
    print(f"Built LEDGER from broker: {len(LEDGER)} positions. WATCHLIST size: {len(WATCHLIST)}")
    if prev:
        display(pd.DataFrame(prev))
    return LEDGER

LEDGER = build_ledger_from_broker()


In [ ]:
# === Cell 4 — Ledger (CSV) & helpers (no infer dependency) ===
from IPython.display import display
from pathlib import Path
import pandas as pd
import requests
from datetime import datetime
from decimal import Decimal, ROUND_DOWN

# Make sure these exist from Cell 1
assert 'TRADING_BASE' in globals() and 'APCA_HEADERS' in globals(), "Run Cell 1 first."

# -------------------- LEDGER storage --------------------
LEDGER_PATH = Path('./out/positions_ledger.csv')
LEDGER_PATH.parent.mkdir(parents=True, exist_ok=True)

# Load/create LEDGER
if 'LEDGER' not in globals():
    if LEDGER_PATH.exists():
        LEDGER = pd.read_csv(LEDGER_PATH)
    else:
        LEDGER = pd.DataFrame(columns=[
            'symbol','entry_time_utc','shares','entry_px','peak_px',
            'stop_pct','trail_pct','tp_pct'
        ])

# -------------------- Fractional safety + asset info --------------------
FRACTION_DECIMALS = 6 
_EXP = Decimal('0.' + ('0'*(FRACTION_DECIMALS-1)) + '1')  # 0.000001

def _round_down_shares(x: float) -> str | None:
    """
    Return a string share qty rounded DOWN to 6 decimals.
    If result is <= 0, return None to avoid 422s.
    """
    d = Decimal(str(x))
    q = d.quantize(_EXP, rounding=ROUND_DOWN)
    if q <= 0:
        return None
    # shave one micro-share when equal to avoid boundary race with 'available'
    if q == d and q > _EXP:
        q = q - _EXP
    return format(q, 'f')

_ASSET_CACHE = {}
def is_fractionable(sym: str) -> bool:
    sym = sym.upper()
    if sym in _ASSET_CACHE:
        return _ASSET_CACHE[sym].get('fractionable', False)
    try:
        r = requests.get(f"{TRADING_BASE}/v2/assets/{sym}", headers=APCA_HEADERS, timeout=5)
        if r.status_code == 200:
            _ASSET_CACHE[sym] = r.json()
            return bool(_ASSET_CACHE[sym].get('fractionable', False))
    except Exception:
        pass
    return False  # safe default

#sync any new purchases in guardian run
def sync_additions_from_broker() -> int:
    df = _broker_positions_df()  # must return ['symbol','qty','avg_entry_price']
    if df.empty:
        return 0
    have = set(LEDGER['symbol'].str.upper())
    new_rows = df[~df['symbol'].str.upper().isin(have)]
    added = 0
    for _, r in new_rows.iterrows():
        qty = float(r['qty'])
        if qty <= 0:
            continue
        record_entry(
            symbol=str(r['symbol']).upper(),
            shares=qty,
            entry_px=float(r['avg_entry_price'])
        )
        added += 1
    return added


# -------------------- Broker-facing helpers --------------------
def broker_position_qty(sym: str) -> float:
    try:
        r = requests.get(f"{TRADING_BASE}/v2/positions/{sym}", headers=APCA_HEADERS, timeout=5)
        if r.status_code != 200:
            return 0.0
        return float(r.json().get('qty', 0))
    except Exception:
        return 0.0

def _broker_positions_df() -> pd.DataFrame:
    r = requests.get(f"{TRADING_BASE}/v2/positions", headers=APCA_HEADERS, timeout=10)
    if r.status_code == 404:
        return pd.DataFrame(columns=['symbol','qty','avg_entry_price'])
    r.raise_for_status()
    js = r.json()
    if not js:
        return pd.DataFrame(columns=['symbol','qty','avg_entry_price'])
    df = pd.DataFrame(js)
    df['qty'] = df['qty'].astype(float)
    df['avg_entry_price'] = df['avg_entry_price'].astype(float)
    return df[['symbol','qty','avg_entry_price']]

def cancel_guard_orders(sym: str) -> int:
    """
    Cancel any open orders for the given symbol (avoid conflicts when exiting).
    Returns count of cancellations attempted.
    """
    try:
        r = requests.get(f"{TRADING_BASE}/v2/orders?status=open", headers=APCA_HEADERS, timeout=10)
        r.raise_for_status()
        open_orders = [o for o in r.json() if str(o.get('symbol','')).upper() == sym.upper()]
        for o in open_orders:
            try:
                requests.delete(f"{TRADING_BASE}/v2/orders/{o['id']}", headers=APCA_HEADERS, timeout=10)
            except Exception:
                pass
        return len(open_orders)
    except Exception:
        return 0

def close_position(sym: str) -> tuple[bool, str]:
    """
    Try to close the entire position via Alpaca's close endpoint.
    Returns (ok, info_or_error_text).
    """
    try:
        r = requests.delete(f"{TRADING_BASE}/v2/positions/{sym}", headers=APCA_HEADERS, timeout=10)
        if r.status_code == 200:
            return True, "closed via /v2/positions/{sym}"
        return False, f"{r.status_code} {r.text}"
    except Exception as e:
        return False, str(e)

def submit_market_sell(sym: str, qty_str: str):
    payload = {'symbol': sym, 'side': 'sell', 'type': 'market', 'time_in_force': 'day', 'qty': qty_str}
    r = requests.post(f"{TRADING_BASE}/v2/orders", headers=APCA_HEADERS, json=payload, timeout=10)
    r.raise_for_status()
    return r.json()

# -------------------- Ledger mutations --------------------
def record_entry(symbol: str, shares: float, entry_px: float,
                 stop_pct: float | None = None,
                 trail_pct: float | None = None,
                 tp_pct: float | None = None):
    """
    Add/replace a symbol in the LEDGER with risk params.
    If a param is None, use the global defaults above.
    """
    global LEDGER
    stop_pct  = HARD_STOP_PCT if stop_pct  is None else float(stop_pct)
    trail_pct = TRAIL_FROM_PEAK_PCT if trail_pct is None else float(trail_pct)
    tp_pct    = TP_PCT if tp_pct    is None else float(tp_pct)

    rec = {
        'symbol': symbol.upper(),
        'entry_time_utc': datetime.utcnow().isoformat(timespec='seconds'),
        'shares': float(shares),
        'entry_px': float(entry_px),
        'peak_px': float(entry_px),  # start peak at entry; we’ll lift it on updates
        'stop_pct': float(stop_pct),
        'trail_pct': float(trail_pct),
        'tp_pct': float(tp_pct),
    }
    # dedupe symbol then append
    LEDGER = LEDGER[~LEDGER['symbol'].eq(rec['symbol'])].reset_index(drop=True)
    LEDGER = pd.concat([LEDGER, pd.DataFrame([rec])], ignore_index=True)
    LEDGER.to_csv(LEDGER_PATH, index=False)
    print(f"[entry] {rec['symbol']}: shares={shares}, entry={entry_px:.4f}, "
          f"hard_stop={entry_px*(1-stop_pct):.4f}, tp={entry_px*(1+tp_pct):.4f}")
    return rec

def update_peak(symbol: str, new_price: float):
    global LEDGER
    i = LEDGER.index[LEDGER['symbol'].eq(symbol.upper())]
    if len(i) == 0:
        return
    idx = i[0]
    prev = float(LEDGER.at[idx, 'peak_px'])
    if new_price > prev:
        LEDGER.at[idx, 'peak_px'] = float(new_price)
        LEDGER.at[idx, 'entry_time_utc'] = datetime.utcnow().isoformat(timespec='seconds')
        LEDGER.to_csv(LEDGER_PATH, index=False)

def current_thresholds(symbol: str) -> dict | None:
    r = LEDGER.loc[LEDGER['symbol'].eq(symbol.upper())]
    if r.empty:
        return None
    row = r.iloc[0]
    entry = float(row['entry_px']); peak = float(row['peak_px'])
    stop  = float(row['stop_pct']);  trail = float(row['trail_pct']); tp = float(row['tp_pct'])
    return {
        'symbol': row['symbol'],
        'entry_px': entry,
        'peak_px': peak,
        'hard_stop_px': round(entry * (1 - stop), 4),
        'trail_stop_px': round(peak  * (1 - trail), 4),
        'tp_px': round(entry * (1 + tp), 4),
        'stop_pct': stop,
        'trail_pct': trail,
        'tp_pct': tp,
    }

def bootstrap_ledger_from_broker():
    """
    Set LEDGER from current Alpaca positions using global risk knobs.
    Keeps the CSV in sync, replacing rows for held symbols.
    """
    df = _broker_positions_df()
    if df.empty:
        print('No broker positions found; LEDGER unchanged.')
        return df

    for _, r in df.iterrows():
        record_entry(
            symbol=str(r['symbol']).upper(),
            shares=float(r['qty']),
            entry_px=float(r['avg_entry_price']),
        )
    print(f'Bootstrapped {len(df)} positions into LEDGER.')

    # quick preview
    prev = []
    for _, row in LEDGER.head(12).iterrows():
        t = current_thresholds(row['symbol'])
        prev.append({k: t[k] for k in ('symbol','entry_px','peak_px','hard_stop_px','trail_stop_px','tp_px')})
    if prev:
        display(pd.DataFrame(prev))
    return df

def reconcile_ledger_with_broker():
    """
    Keep LEDGER rows aligned with actual broker holdings (qty updates, add/remove).
    """
    global LEDGER
    df = _broker_positions_df()
    held = set(df['symbol'].str.upper()) if not df.empty else set()

    # prune symbols no longer held
    before = len(LEDGER)
    LEDGER = LEDGER[LEDGER['symbol'].isin(held)].reset_index(drop=True)

    # ensure entries exist / update share qty to match broker
    idx_map = {s:i for i,s in enumerate(LEDGER['symbol'])}
    for _, r in df.iterrows():
        sym = str(r['symbol']).upper()
        qty = float(r['qty']); aep = float(r['avg_entry_price'])
        if sym in idx_map:
            i = idx_map[sym]
            LEDGER.at[i, 'shares'] = qty
            # seed entry/peak if missing or non-positive
            if pd.isna(LEDGER.at[i, 'entry_px']) or float(LEDGER.at[i, 'entry_px']) <= 0:
                LEDGER.at[i, 'entry_px'] = aep
                LEDGER.at[i, 'peak_px'] = aep
        else:
            record_entry(sym, qty, aep)

    LEDGER.to_csv(LEDGER_PATH, index=False)
    return held

# Show tail for quick sanity check
LEDGER.tail(20)

# --- put this right after you load/create LEDGER in Cell 4 ---
# Ensure stable column set and dtypes
want_cols = ['symbol','entry_time_utc','shares','entry_px','peak_px','stop_pct','trail_pct','tp_pct']
for c in want_cols:
    if c not in LEDGER.columns:
        LEDGER[c] = pd.Series(dtype='object')

# force types
LEDGER['symbol'] = LEDGER['symbol'].astype(str)
LEDGER['entry_time_utc'] = LEDGER['entry_time_utc'].astype(str)
for c in ['shares','entry_px','peak_px','stop_pct','trail_pct','tp_pct']:
    LEDGER[c] = pd.to_numeric(LEDGER[c], errors='coerce')



In [ ]:
# === Cell 4.5 — Ensure/Migrate LEDGER schema (run once if needed) ===
import numpy as np

REQUIRED = ['symbol','entry_time_utc','shares','entry_px','peak_px','stop_pct','trail_pct','tp_pct']

def _ensure_ledger_schema():
    global LEDGER
    changed = False

    # Add any missing required columns
    for col in REQUIRED:
        if col not in LEDGER.columns:
            LEDGER[col] = np.nan
            changed = True

    # Back-calc percentages if older absolute cols exist
    if 'hard_stop_px' in LEDGER.columns and LEDGER['stop_pct'].isna().any():
        with np.errstate(divide='ignore', invalid='ignore'):
            LEDGER['stop_pct'] = np.where(
                LEDGER['stop_pct'].isna() & (pd.to_numeric(LEDGER['entry_px'], errors='coerce') > 0),
                (pd.to_numeric(LEDGER.get('entry_px'), errors='coerce') -
                 pd.to_numeric(LEDGER.get('hard_stop_px'), errors='coerce')) /
                 pd.to_numeric(LEDGER.get('entry_px'), errors='coerce'),
                LEDGER['stop_pct']
            )
        changed = True

    if 'tp_px' in LEDGER.columns and LEDGER['tp_pct'].isna().any():
        with np.errstate(divide='ignore', invalid='ignore'):
            LEDGER['tp_pct'] = np.where(
                LEDGER['tp_pct'].isna() & (pd.to_numeric(LEDGER['entry_px'], errors='coerce') > 0),
                (pd.to_numeric(LEDGER.get('tp_px'), errors='coerce') -
                 pd.to_numeric(LEDGER.get('entry_px'), errors='coerce')) /
                 pd.to_numeric(LEDGER.get('entry_px'), errors='coerce'),
                LEDGER['tp_pct']
            )
        changed = True

    if 'trail_stop_px' in LEDGER.columns and LEDGER['trail_pct'].isna().any():
        with np.errstate(divide='ignore', invalid='ignore'):
            LEDGER['trail_pct'] = np.where(
                LEDGER['trail_pct'].isna() & (pd.to_numeric(LEDGER['peak_px'], errors='coerce') > 0),
                (pd.to_numeric(LEDGER.get('peak_px'), errors='coerce') -
                 pd.to_numeric(LEDGER.get('trail_stop_px'), errors='coerce')) /
                 pd.to_numeric(LEDGER.get('peak_px'), errors='coerce'),
                LEDGER['trail_pct']
            )
        changed = True

    # Fill any remaining NaNs with current global defaults from Cell 4
    for c, default in [('stop_pct', HARD_STOP_PCT),
                       ('trail_pct', TRAIL_FROM_PEAK_PCT),
                       ('tp_pct', TP_PCT)]:
        if LEDGER[c].isna().any():
            LEDGER[c] = LEDGER[c].fillna(default)
            changed = True

    # Ensure numeric types and basic sanity
    for c in ['shares','entry_px','peak_px','stop_pct','trail_pct','tp_pct']:
        LEDGER[c] = pd.to_numeric(LEDGER[c], errors='coerce')
    LEDGER['peak_px'] = LEDGER['peak_px'].fillna(LEDGER['entry_px'])

    # Reorder + persist if anything changed or order is wrong
    LEDGER = LEDGER[REQUIRED]
    if changed:
        LEDGER.to_csv(LEDGER_PATH, index=False)
        print(f"[schema] Upgraded ledger schema and saved ({len(LEDGER)} rows).")
    else:
        print("[schema] Ledger schema already OK.")

_ensure_ledger_schema()
print("Columns now:", list(LEDGER.columns))
display(LEDGER.head(10))


In [ ]:
# === Cell 5 — Price, order helpers, reconcile, RTH gate, logging (+ Alpaca action audit & dust sweep) ===
from pathlib import Path
import uuid, requests, csv, os, json
from datetime import datetime

# ---------- Price helpers ----------
def last_px(symbol: str):
    """
    Latest tradable price priority:
      1) snapshot.latestTrade.p
      2) snapshot.MinuteBar.c
      3) last 1-minute bar close
    """
    sym = symbol.upper()
    try:
        r = requests.get(
            f"{DATA_BASE}/v2/stocks/{sym}/snapshot",
            headers=APCA_HEADERS, params={'feed': 'iex'}, timeout=10
        )
        if r.status_code == 200:
            js = r.json() or {}
            latest_trade = (js.get('latestTrade') or {}).get('p')
            minute_bar_c = (js.get('MinuteBar') or {}).get('c')
            if latest_trade is not None:
                return float(latest_trade)
            if minute_bar_c is not None:
                return float(minute_bar_c)
    except Exception:
        pass

    try:
        rr = requests.get(
            f"{DATA_BASE}/v2/stocks/{sym}/bars",
            headers=APCA_HEADERS,
            params={'timeframe': '1Min', 'limit': 1, 'feed': 'iex'},
            timeout=10
        )
        rr.raise_for_status()
        bars = rr.json().get('bars', [])
        return float(bars[-1]['c']) if bars else None
    except Exception:
        return None


def last_minute_low(symbol: str):
    """Low of the last 1-minute bar (helps catch intraminute breaches)."""
    sym = symbol.upper()
    try:
        rr = requests.get(
            f"{DATA_BASE}/v2/stocks/{sym}/bars",
            headers=APCA_HEADERS,
            params={'timeframe': '1Min', 'limit': 1, 'feed': 'iex'},
            timeout=10
        )
        if rr.status_code == 200:
            bars = rr.json().get('bars', [])
            if bars:
                return float(bars[-1]['l'])
    except Exception:
        pass
    return None


# ---------- Strategy-level log (guardian decisions) ----------
ACTIONS_LOG = './out/actions_log.csv'
Path('./out').mkdir(parents=True, exist_ok=True)

def log_action(row: dict):
    write_header = not os.path.exists(ACTIONS_LOG)
    with open(ACTIONS_LOG, 'a', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=sorted(row.keys()))
        if write_header:
            w.writeheader()
        w.writerow(row)


# ---------- Alpaca action audit (API requests/responses) ----------
ALPACA_ACTIONS_LOG = './out/alpaca_actions.csv'

def _append_csv_row(path: str, row: dict):
    write_header = not os.path.exists(path)
    with open(path, 'a', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=sorted(row.keys()))
        if write_header:
            w.writeheader()
        w.writerow(row)

def log_alpaca_action(
    event: str,
    symbol: str,
    payload: dict | None = None,
    response: dict | None = None,
    error: dict | str | None = None,
    context: str = 'guardian'
):
    """
    event: 'order_submit_ok' | 'order_submit_error' | 'order_cancel_ok' | 'order_cancel_error'
           'position_close_ok' | 'position_close_status' | 'position_close_error' | 'order_cancel_status' | etc.
    """
    row = {
        'ts': datetime.utcnow().isoformat(timespec='seconds'),
        'context': context,
        'event': event,
        'symbol': symbol.upper() if symbol else None,
    }
    if payload:
        row.update({k: payload.get(k) for k in ('side','type','time_in_force','qty','client_order_id','why','symbol','limit_price','extended_hours') if k in payload})
    if response:
        # keep a concise view + raw JSON for any debugging later
        row.update({
            'order_id': response.get('id'),
            'status': response.get('status') if 'status' in response else response.get('code'),
            'filled_qty': response.get('filled_qty'),
            'filled_avg_price': response.get('filled_avg_price'),
            'submitted_at': response.get('submitted_at'),
        })
        try:
            row['raw_response'] = json.dumps(response, default=str)
        except Exception:
            row['raw_response'] = str(response)
    if error is not None:
        if isinstance(error, dict):
            for k, v in error.items():
                row[f'err_{k}'] = v
        else:
            row['err_text'] = str(error)

    _append_csv_row(ALPACA_ACTIONS_LOG, row)


# ---------- Order helpers (with audit logging) ----------
def submit_order(payload: dict):
    """POST /v2/orders + audit log."""
    p = dict(payload)
    p.setdefault('client_order_id', f"guard-{uuid.uuid4().hex[:12]}")
    r = requests.post(f"{TRADING_BASE}/v2/orders", headers=APCA_HEADERS, json=p, timeout=10)

    if r.status_code >= 400:
        log_alpaca_action(
            event='order_submit_error',
            symbol=p.get('symbol', ''),
            payload=p,
            error={'status_code': r.status_code, 'body': r.text}
        )
        print('[order error]', r.status_code, r.text)
        r.raise_for_status()

    data = r.json()
    log_alpaca_action(event='order_submit_ok', symbol=p.get('symbol', ''), payload=p, response=data)
    return data


# --- Replace your cancel_guard_orders with this version (Cell 5) ---
def _order_brief(o: dict) -> dict:
    return {
        'id': o.get('id'),
        'symbol': o.get('symbol'),
        'side': o.get('side'),
        'type': o.get('type'),
        'time_in_force': o.get('time_in_force'),
        'status': o.get('status'),
        'submitted_at': o.get('submitted_at'),
        'created_at': o.get('created_at'),
        'filled_qty': o.get('filled_qty'),
        'filled_avg_price': o.get('filled_avg_price'),
        'client_order_id': o.get('client_order_id'),
    }

def cancel_guard_orders(symbol: str, why: str = "pre-exit"):
    """
    Cancel open guardian-created orders for `symbol`.
    We log a pre-cancel snapshot and the reason (`why`).
    """
    sym = symbol.upper()
    try:
        r = requests.get(f"{TRADING_BASE}/v2/orders",
                         headers=APCA_HEADERS,
                         params={'status': 'open'},
                         timeout=10)
        r.raise_for_status()
        for o in r.json():
            if o.get('symbol','').upper() == sym and str(o.get('client_order_id','')).startswith('guard-'):
                pre = _order_brief(o)
                rr = requests.delete(f"{TRADING_BASE}/v2/orders/{o['id']}",
                                     headers=APCA_HEADERS, timeout=10)
                evt = 'order_cancel_ok' if rr.status_code in (200, 204) else 'order_cancel_status'
                log_alpaca_action(
                    event=evt,
                    symbol=sym,
                    payload={'client_order_id': o.get('client_order_id'), 'why': why},
                    response={'status': rr.status_code, 'order_id': o.get('id'), 'pre': pre}
                )
    except Exception as e:
        log_alpaca_action(event='order_cancel_error', symbol=sym,
                          error={'why': why, 'text': str(e)})
        print(f"[cancel] {sym}: {e}")


def close_position(symbol: str) -> tuple[bool, str]:
    """
    Force-close the full position via Alpaca (handles whatever is available).
    Useful fallback if a market sell hits fractional precision issues.
    """
    sym = symbol.upper()
    try:
        rr = requests.delete(f"{TRADING_BASE}/v2/positions/{sym}", headers=APCA_HEADERS, timeout=10)
        ok = (rr.status_code == 200)
        if ok:
            log_alpaca_action(event='position_close_ok', symbol=sym, response={'status': rr.status_code})
            return True, "closed via DELETE /v2/positions/{sym}"
        else:
            log_alpaca_action(event='position_close_status', symbol=sym, response={'status': rr.status_code, 'body': rr.text})
            return False, f"{rr.status_code} {rr.text}"
    except Exception as e:
        log_alpaca_action(event='position_close_error', symbol=sym, error=str(e))
        return False, str(e)


# ---------- Dust handling & full-symbol cancel (for sub-min micro-shares) ----------
EPS_SHARES = 1e-6   # ≤ 0.000001 share is dust
EPS_USD    = 0.05   # or ≤ 5 cents notional is dust

def is_dust(sym: str, qty: float) -> bool:
    px = last_px(sym) or 0.0
    return (qty <= EPS_SHARES) or (px * qty <= EPS_USD)

def cancel_all_orders_for_symbol(symbol: str):
    """Cancel ALL open orders for the symbol (not just guardian-tagged)."""
    sym = symbol.upper()
    try:
        r = requests.get(f"{TRADING_BASE}/v2/orders", headers=APCA_HEADERS,
                         params={'status': 'open'}, timeout=10)
        r.raise_for_status()
        for o in r.json():
            if o.get('symbol','').upper() == sym:
                rr = requests.delete(f"{TRADING_BASE}/v2/orders/{o['id']}", headers=APCA_HEADERS, timeout=10)
                evt = 'order_cancel_ok' if rr.status_code in (200, 204) else 'order_cancel_status'
                log_alpaca_action(
                    event=evt, symbol=sym,
                    payload={'client_order_id': o.get('client_order_id')},
                    response={'id': o.get('id'), 'status': rr.status_code, 'body': rr.text}
                )
    except Exception as e:
        log_alpaca_action(event='order_cancel_error', symbol=sym, error=str(e))
        print(f"[cancel_all] {sym}: {e}")

def sweep_dust(symbol: str) -> tuple[bool, str]:
    """
    Best-effort clean-up for sub-min micro-shares:
      1) Cancel ALL open orders for the symbol (free up held qty).
      2) Re-check position; if dust, try DELETE /v2/positions/{sym}.
      3) If broker still refuses (e.g., held=0.000000149), log DUST_IGNORE.
    Returns (ok, message). ok=True means no further action needed.
    """
    sym = symbol.upper()
    try:
        cancel_all_orders_for_symbol(sym)

        rp = requests.get(f"{TRADING_BASE}/v2/positions/{sym}", headers=APCA_HEADERS, timeout=10)
        if rp.status_code != 200:
            log_action({'ts': datetime.utcnow().isoformat(timespec='seconds'), 'event': 'DUST_SWEEP', 'symbol': sym, 'ok': True, 'msg': 'no position'})
            return True, "no position"

        pos = rp.json() or {}
        qty = float(pos.get('qty', 0.0))
        if qty <= 0:
            log_action({'ts': datetime.utcnow().isoformat(timespec='seconds'), 'event': 'DUST_SWEEP', 'symbol': sym, 'ok': True, 'msg': 'qty 0 after cancel'})
            return True, "qty 0 after cancel"

        if not is_dust(sym, qty):
            return False, f"not dust (qty={qty})"

        rr = requests.delete(f"{TRADING_BASE}/v2/positions/{sym}", headers=APCA_HEADERS, timeout=10)
        if rr.status_code == 200:
            log_action({'ts': datetime.utcnow().isoformat(timespec='seconds'), 'event': 'DUST_SWEEP', 'symbol': sym, 'ok': True, 'msg': 'closed via DELETE', 'qty': qty})
            log_alpaca_action(event='position_close_ok', symbol=sym, response={'status': rr.status_code})
            return True, "closed via DELETE"

        # 403 insufficient qty available => treat as dust and ignore
        body = rr.text
        if rr.status_code == 403 and 'insufficient qty available' in (body or '').lower():
            log_action({'ts': datetime.utcnow().isoformat(timespec='seconds'), 'event': 'DUST_IGNORE', 'symbol': sym, 'ok': True, 'msg': body, 'qty': qty})
            log_alpaca_action(event='position_close_status', symbol=sym, response={'status': rr.status_code, 'body': body})
            return True, "ignored sub-min dust"

        log_alpaca_action(event='position_close_status', symbol=sym, response={'status': rr.status_code, 'body': body})
        return False, f"{rr.status_code} {body}"

    except Exception as e:
        log_action({'ts': datetime.utcnow().isoformat(timespec='seconds'), 'event': 'DUST_SWEEP', 'symbol': sym, 'ok': False, 'msg': str(e)})
        log_alpaca_action(event='position_close_error', symbol=sym, error=str(e))
        return False, str(e)


# ---------- Broker reconcile & RTH gate ----------
def broker_symbols() -> set[str]:
    r = requests.get(f"{TRADING_BASE}/v2/positions", headers=APCA_HEADERS, timeout=10)
    if r.status_code == 404:
        return set()
    r.raise_for_status()
    js = r.json()
    if isinstance(js, dict):
        js = js.get('positions', [])
    return {str(p['symbol']).upper() for p in js}

def reconcile_ledger_with_broker():
    """
    Keep LEDGER rows only for currently-held broker symbols.
    Requires LEDGER/LEDGER_PATH from Cell 4.
    """
    global LEDGER
    held = broker_symbols()
    before = len(LEDGER)
    LEDGER = LEDGER[LEDGER['symbol'].isin(held)].reset_index(drop=True)
    if len(LEDGER) != before:
        LEDGER.to_csv(LEDGER_PATH, index=False)

def within_rth(now_local) -> bool:
    t = now_local.timetz().replace(tzinfo=None)
    return MARKET_START_PT <= t <= MARKET_END_PT


# --- Order/position utilities for dust + tick rounding (Cell 5) ---
from decimal import Decimal, ROUND_DOWN

def fetch_open_orders(symbol: str | None = None):
    r = requests.get(f"{TRADING_BASE}/v2/orders", headers=APCA_HEADERS,
                     params={'status': 'open'}, timeout=10)
    r.raise_for_status()
    orders = r.json()
    if symbol:
        sym = symbol.upper()
        orders = [o for o in orders if str(o.get('symbol','')).upper() == sym]
    return orders

def cancel_all_symbol_orders(symbol: str):
    """Cancel ALL open orders for symbol (not just guard-ones)."""
    sym = symbol.upper()
    try:
        for o in fetch_open_orders(sym):
            rr = requests.delete(f"{TRADING_BASE}/v2/orders/{o['id']}", headers=APCA_HEADERS, timeout=10)
            log_alpaca_action(
                event=('order_cancel_ok' if rr.status_code in (200,204) else 'order_cancel_status'),
                symbol=sym,
                payload={'client_order_id': o.get('client_order_id'), 'origin':'any'},
                response={'id': o.get('id'), 'status': rr.status_code, 'body': rr.text}
            )
    except Exception as e:
        log_alpaca_action(event='order_cancel_error', symbol=sym, error=str(e))

def get_position_status(symbol: str):
    """Return dict with qty, held_for_orders, available (floats)."""
    sym = symbol.upper()
    r = requests.get(f"{TRADING_BASE}/v2/positions/{sym}", headers=APCA_HEADERS, timeout=10)
    if r.status_code != 200:
        return {'qty':0.0,'held_for_orders':0.0,'available':0.0}
    js = r.json()
    def _f(k): 
        try: return float(js.get(k, 0))
        except: return 0.0
    return {
        'qty': _f('qty'),
        'held_for_orders': _f('held_for_orders'),
        'available': _f('available'),
    }

def wait_release(symbol: str, timeout_sec: int = 5):
    """Wait briefly for held_for_orders to clear after cancels."""
    import time
    t0 = time.time()
    while time.time() - t0 < timeout_sec:
        st = get_position_status(symbol)
        if st['held_for_orders'] <= 0:
            return True
        time.sleep(0.4)
    return False

def tick_round(price: float) -> float:
    """US equities: >=$1 use $0.01; <$1 use $0.0001 (rounded DOWN)."""
    if price >= 1:
        return float(Decimal(str(price)).quantize(Decimal('0.01'), rounding=ROUND_DOWN))
    else:
        return float(Decimal(str(price)).quantize(Decimal('0.0001'), rounding=ROUND_DOWN))

# ---------- NEW: Quotes & tick helpers used by Cell 6 for EXT orders ----------
def _best_bid(symbol: str) -> float | None:
    """
    Try latest quote endpoint first, then snapshot.latestQuote.bp, else None.
    """
    sym = symbol.upper()
    # 1) Latest quote (preferred)
    try:
        rq = requests.get(
            f"{DATA_BASE}/v2/stocks/{sym}/quotes/latest",
            headers=APCA_HEADERS,
            params={'feed':'iex'},
            timeout=6
        )
        if rq.status_code == 200:
            jq = rq.json() or {}
            q  = jq.get('quote') or {}
            bp = q.get('bp')
            if bp is not None:
                return float(bp)
    except Exception:
        pass
    # 2) Snapshot fallback
    try:
        rs = requests.get(
            f"{DATA_BASE}/v2/stocks/{sym}/snapshot",
            headers=APCA_HEADERS,
            params={'feed':'iex'},
            timeout=6
        )
        if rs.status_code == 200:
            js = rs.json() or {}
            lq = js.get('latestQuote') or {}
            bp = lq.get('bp')
            if bp is not None:
                return float(bp)
    except Exception:
        pass
    return None

def _tick_for_price(price: float) -> float:
    """Return the minimum tick size for a given price."""
    return 0.01 if price >= 1.0 else 0.0001

def _floor_to_tick(price: float, tick: float) -> float:
    """
    Floor the price to the provided tick, using Decimal to avoid fp drift.
    """
    if tick <= 0:
        tick = 0.01
    # scale to integer ticks, floor, then rescale
    q = int(Decimal(str(price)) / Decimal(str(tick)))
    return float(Decimal(q) * Decimal(str(tick)))

def _fmt_price(price: float) -> str:
    """
    Format a price string matching the tick precision (2dp for >=$1, 4dp for <$1).
    """
    if price >= 1.0:
        return f"{price:.2f}"
    else:
        return f"{price:.4f}"

# --- Market calendar & phase helpers (used by Cell 6) ---
from dateutil import tz as _tz

# Reuse TZ from Cell 1 if present; otherwise set fallbacks
_ET = _tz.gettz('America/New_York')
_TZ = globals().get('TZ') or _tz.gettz('America/Los_Angeles')

# Pull session bounds from globals (if you set them earlier)
_PRE_START_PT = globals().get('PRE_START_PT')   # e.g., dtime(1, 0)   # 04:00 ET
_RTH_START_PT = globals().get('RTH_START_PT')   # e.g., dtime(6, 30)  # 09:30 ET
_RTH_END_PT   = globals().get('RTH_END_PT')     # e.g., dtime(13, 0)  # 16:00 ET
_POST_END_PT  = globals().get('POST_END_PT')    # e.g., dtime(17, 0)  # 20:00 ET

def is_trading_day_today() -> bool:
    """Use Alpaca calendar to confirm today is a trading day (holiday/weekend safe)."""
    d_et = datetime.now(_ET).date().isoformat()
    try:
        r = requests.get(f"{TRADING_BASE}/v2/calendar",
                         headers=APCA_HEADERS,
                         params={'start': d_et, 'end': d_et},
                         timeout=8)
        if r.status_code != 200:
            return False
        return len(r.json() or []) > 0
    except Exception:
        return False

def market_phase(now_local) -> str:
    """
    Returns: 'rth' | 'pre' | 'post' | 'closed'
    - 'closed' covers weekends/holidays and off-hours outside pre/post ranges.
    """
    # Not a trading day at all
    if not is_trading_day_today():
        return 'closed'
    # If you didn't define session bounds upstream, fall back to 'closed'
    if not (_RTH_START_PT and _RTH_END_PT and _PRE_START_PT and _POST_END_PT):
        return 'closed'

    t = now_local.timetz().replace(tzinfo=None)
    in_rth  = _RTH_START_PT <= t <= _RTH_END_PT
    in_pre  = _PRE_START_PT <= t <  _RTH_START_PT
    in_post = _RTH_END_PT   <  t <= _POST_END_PT
    if in_rth:  return 'rth'
    if in_pre:  return 'pre'
    if in_post: return 'post'
    return 'closed'



In [ ]:
# === Cell 6 — Guardian (price checks + exits + dust sweep + auto-sync adds) ===
from datetime import datetime, timedelta

# ---- knobs (override upstream if you want) ----
GUARD_INTERVAL_MIN = globals().get('GUARD_INTERVAL_MIN', 2)    # run every N minutes
EPS_SHARES         = globals().get('EPS_SHARES', 1e-6)         # "effectively zero" shares

# ---- pull helpers from prior cells (soft refs so renames won't crash) ----
_last_px             = globals().get('last_px')
_last_minute_low     = globals().get('last_minute_low')
_submit_order        = globals().get('submit_order')
_cancel_guard_orders = globals().get('cancel_guard_orders')
_close_position      = globals().get('close_position')
_reconcile           = globals().get('reconcile_ledger_with_broker')
_within_session      = globals().get('within_session')     # uses SESSION_MODE, PRE/RTH/POST bounds
_is_fractionable     = globals().get('is_fractionable')
_broker_qty          = globals().get('broker_position_qty')
_round_down_shares   = globals().get('_round_down_shares') # from Cell 4
_best_bid            = globals().get('_best_bid')          # from Cell 5
_tick_for_price      = globals().get('_tick_for_price')    # from Cell 5
_floor_to_tick       = globals().get('_floor_to_tick')     # from Cell 5
_fmt_price           = globals().get('_fmt_price')         # from Cell 5
_sweep_dust          = globals().get('sweep_dust')         # from Cell 5 (optional)
_market_phase        = globals().get('market_phase') # adding from cell 5 for checking time to include weekend/weekday 
_log_action          = globals().get('log_action')

# RTH window (defined earlier; used here for order-type choice)
RTH_START_PT = globals().get('RTH_START_PT')
RTH_END_PT   = globals().get('RTH_END_PT')

# Fallback TZ (should be set in Cell 1)
TZ = globals().get('TZ')

# Safe peak updater (avoids pandas dtype warning on entry_time_utc)
def _safe_update_peak(symbol: str, new_price: float):
    global LEDGER
    sym = symbol.upper()
    i = LEDGER.index[LEDGER['symbol'].astype(str).str.upper().eq(sym)]
    if len(i) == 0:
        return
    idx = i[0]
    prev = float(LEDGER.at[idx, 'peak_px'])
    if new_price > prev:
        LEDGER.at[idx, 'peak_px'] = float(new_price)
        # ensure object dtype for timestamp column
        if 'entry_time_utc' in LEDGER.columns and LEDGER['entry_time_utc'].dtype != 'O':
            LEDGER['entry_time_utc'] = LEDGER['entry_time_utc'].astype('object')
        LEDGER.at[idx, 'entry_time_utc'] = datetime.utcnow().isoformat(timespec='seconds')
        LEDGER.to_csv(LEDGER_PATH, index=False)

def _is_rth(now_local) -> bool:
    if RTH_START_PT is None or RTH_END_PT is None:
        return True  # if not defined upstream, default to "treat as RTH"
    t = now_local.timetz().replace(tzinfo=None)
    return RTH_START_PT <= t <= RTH_END_PT

# ---- auto-add new Alpaca positions into LEDGER (uses Cell 4 record_entry) ----
def sync_additions_from_broker() -> int:
    if 'record_entry' not in globals() or '_broker_positions_df' not in globals():
        return 0
    df = _broker_positions_df()
    if df.empty:
        return 0
    tracked = set(LEDGER['symbol'].astype(str).str.upper())
    added = 0
    for _, r in df.iterrows():
        sym = str(r['symbol']).upper()
        if sym in tracked:
            continue
        record_entry(
            symbol=sym,
            shares=float(r['qty']),
            entry_px=float(r['avg_entry_price']),
        )
        added += 1
    return added

# ---- main pass: check thresholds and exit when breached ----
async def guard_once(verbose: bool = True):
    global LEDGER
    ts = datetime.now(TZ).strftime('%H:%M:%S')

    # Time gating (RTH/EXT/ALWAYS) via within_session from Cell 6 Session block
    if _within_session and not _within_session(datetime.now(TZ)):
        if verbose:
            print(f"[guardian] {ts} outside allowed session; skipping")
        return

    # Pull in any newly-bought symbols we weren't tracking
    added = sync_additions_from_broker()
    if added and verbose:
        print(f"[guardian] synced {added} new positions from broker.")

    # Keep LEDGER aligned to what's actually held
    if _reconcile:
        _reconcile()

    if LEDGER.empty:
        if verbose:
            print(f"[guardian] {ts} checked 0/0 | exits=0 (ledger empty)")
        return

    total, checked, exits = len(LEDGER), 0, []
    for _, r in LEDGER.copy().iterrows():
        sym = str(r["symbol"]).upper(); checked += 1

        # Prices: latest trade + last minute low (use lower for stops)
        px  = _last_px(sym) if _last_px else None
        low = _last_minute_low(sym) if _last_minute_low else None
        eff = None
        if px is not None and low is not None:
            eff = min(px, low)
        elif px is not None:
            eff = px
        elif low is not None:
            eff = low
        if eff is None:
            continue

        # Maintain trailing peak (safe)
        _safe_update_peak(sym, px if px is not None else eff)

        th = current_thresholds(sym)
        if th is None:
            continue

        reasons = []
        if eff <= th["hard_stop_px"]:  reasons.append("hard")
        if eff <= th["trail_stop_px"]: reasons.append("trail")
        if px is not None and px >= th["tp_px"]: reasons.append("tp")
        if not reasons:
            continue

        try:
            # clear any open guardian orders first
            if _cancel_guard_orders:
                _cancel_guard_orders(sym)

            bqty = _broker_qty(sym) if _broker_qty else 0.0
            if bqty <= 0:
                print(f"[skip exit] {sym}: broker qty is 0")
                continue

            # quantity formatting
            if _is_fractionable and _is_fractionable(sym):
                qty_str = _round_down_shares(bqty) if _round_down_shares else None
                if not qty_str:
                    # micro-dust fallback
                    if _close_position:
                        ok, info = _close_position(sym)
                        if not ok:
                            print(f"[close-position fallback failed] {sym}: {info}")
                        # attempt a dust sweep once (best-effort)
                        if _sweep_dust:
                            try: _sweep_dust(sym)
                            except Exception: pass
                        continue
            else:
                q_int = int(bqty)
                if q_int < 1:
                    print(f"[skip exit] {sym}: non-fractionable and <1 share")
                    continue
                qty_str = str(q_int)

            # decide order type: Market in RTH, marketable LIMIT (extended hours) outside RTH
            in_rth = _is_rth(datetime.now(TZ))
            # Decide order type by phase: market in RTH; limit DAY in pre/post; limit GTC when closed
            phase = _market_phase(datetime.now(TZ)) if _market_phase else 'closed'
            if phase == 'rth':
                order = {
                    'symbol': sym, 'side': 'sell', 'type': 'market',
                    'time_in_force': 'day', 'qty': qty_str
                }
            elif phase in ('pre', 'post'):
                # marketable limit at/below best bid, DAY + extended_hours=True
                ref_candidates = [v for v in (
                    (_best_bid(sym) if _best_bid else None),
                    px, low
                ) if v is not None]
                ref  = min(ref_candidates) if ref_candidates else eff
                tick = _tick_for_price(ref) if _tick_for_price else 0.01
                limit = _floor_to_tick(ref - tick, tick) if _floor_to_tick else max(0.01, round(ref - 0.01, 2))
                order = {
                    'symbol': sym, 'side': 'sell', 'type': 'limit',
                    'time_in_force': 'day', 'qty': qty_str,
                    'limit_price': _fmt_price(limit) if _fmt_price else f"{limit:.2f}",
                    'extended_hours': True
                }
            else:
                # Fully closed (weekend/holiday/overnight): queue GTC limit for next session
                ref  = px if px is not None else eff
                tick = _tick_for_price(ref) if _tick_for_price else 0.01
                limit = _floor_to_tick(ref - tick, tick) if _floor_to_tick else max(0.01, round(ref - 0.01, 2))
                order = {
                    'symbol': sym, 'side': 'sell', 'type': 'limit',
                    'time_in_force': 'gtc', 'qty': qty_str,
                    'limit_price': _fmt_price(limit) if _fmt_price else f"{limit:.2f}",
                    'extended_hours': False
                }

            # submit
            if _submit_order:
                _submit_order(order)

            # best-effort dust sweep (harmless if nothing left)
            if _sweep_dust:
                try: _sweep_dust(sym)
                except Exception: pass

            # strategy-level log
            if _log_action:
                _log_action({
                    'ts': datetime.utcnow().isoformat(timespec='seconds'),
                    'event': 'EXIT', 'symbol': sym,
                    'reasons': '/'.join(reasons),
                    'px': px, 'low': low, 'qty': order.get('qty', 'CLOSE_POS'),
                    'ord_type': order.get('type'), 'limit': order.get('limit_price'),
                    'phase': phase
                })

        except Exception as e:
            print(f"[EXIT error] {sym}: {e}")
            continue
        else:
            # Only drop from LEDGER if the broker position is effectively gone
            new_qty = _broker_qty(sym) if _broker_qty else 0.0
            if new_qty <= EPS_SHARES:
                exits.append(f"{sym}({'/'.join(reasons)})")
                LEDGER = LEDGER[~LEDGER['symbol'].eq(sym)].reset_index(drop=True)
                LEDGER.to_csv(LEDGER_PATH, index=False)

    if verbose:
        mode = globals().get('SESSION_MODE', 'RTH/EXT/ALWAYS?')
        msg = (f"✅ guardian tick ({mode}) {ts} checked {checked}/{total} | "
               f"exits={len(exits)}" + (f" -> {', '.join(exits[:6])}" if exits else ""))
        print(msg)

# ---- cadence sleep ----
async def _sleep_to_next_interval():
    now = datetime.now(TZ)
    step = int(GUARD_INTERVAL_MIN)
    next_min = ((now.minute // step) + 1) * step
    next_tick = now.replace(minute=next_min % 60, second=0, microsecond=0)
    if next_min >= 60:
        next_tick += timedelta(hours=1)
    delay = max(5.0, (next_tick - now).total_seconds())
    print(f"[guardian] next check at {next_tick.strftime('%H:%M:%S')} (~{int(delay)}s)")
    await asyncio.sleep(delay)

# ---- loop ----
async def guard_loop():
    cadence = int(GUARD_INTERVAL_MIN)
    mode = globals().get('SESSION_MODE', 'RTH/EXT/ALWAYS?')
    print(f"✅ guardian started (cadence={cadence} min, mode={mode}).")
    while True:
        try:
            await guard_once(verbose=True)
        except Exception as e:
            print('[guard_loop] error:', e)
        await _sleep_to_next_interval()


In [ ]:
# === Cell 7 — Cadence knob + Bootstrap (if needed) + Preview + Recent actions ===
from IPython.display import display
import os
import pandas as pd

# --- Cadence: how often the guardian runs (minutes) ---
# The guardian loop in Cell 6 reads this global.
CADENCE_MIN = globals().get('CADENCE_MIN', 2)  # e.g., 2 means every 2 minutes

def set_cadence(minutes: int):
    """Change the guardian cadence (in minutes)."""
    global CADENCE_MIN
    CADENCE_MIN = max(1, int(minutes))
    msg = f"Cadence set to {CADENCE_MIN} minute(s)."
    if 'guard_task' in globals():
        msg += " If the guardian is already running, stop it (Cell 9) and start again (Cell 8) to apply immediately."
    print(msg)

# --- Preview thresholds for tracked positions (from LEDGER) ---
def preview_thresholds():
    if LEDGER.empty:
        print('LEDGER is empty (no positions tracked yet).')
        return
    rows = []
    for sym in sorted(LEDGER['symbol'].unique()):
        th = current_thresholds(sym)
        if not th:
            continue
        px = last_px(sym)  # from Cell 5
        rows.append({
            'symbol': sym,
            'last_px': px,
            'entry_px': th['entry_px'],
            'peak_px': th['peak_px'],
            'hard_stop_px': th['hard_stop_px'],
            'trail_stop_px': th['trail_stop_px'],
            'tp_px': th['tp_px'],
        })
    if rows:
        print(f"(cadence = {CADENCE_MIN} min) Threshold preview:")
        display(pd.DataFrame(rows))
    else:
        print("No thresholds to preview (check LEDGER contents).")

# --- Show messages/actions the guardian took (from CSV log) ---
def show_recent_actions(n: int = 30):
    """Tail the actions log that guard_once() writes (Cell 5’s ACTIONS_LOG)."""
    if not os.path.exists(ACTIONS_LOG):
        print("No actions_log yet.")
        return
    df = pd.read_csv(ACTIONS_LOG)
    if 'ts' in df.columns:
        df['ts'] = pd.to_datetime(df['ts'], errors='coerce')
        df = df.sort_values('ts')
    print(f"Last {min(n, len(df))} actions:")
    display(df.tail(n).reset_index(drop=True))

# --- One-time bootstrap per session (only if LEDGER is empty), then quick preview ---
if LEDGER.empty:
    bootstrap_ledger_from_broker()

preview_thresholds()
# Optionally show last actions:
# show_recent_actions(50)


In [ ]:
# === Cell 8 — Start guardian ===
# If a previous loop is running, stop it first (prevents double-starts)
if 'guard_task' in globals():
    try:
        guard_task.cancel()
        await asyncio.gather(guard_task, return_exceptions=True)
    except Exception:
        pass

guard_task = asyncio.create_task(guard_loop())
print(f"Guardian task created (cadence={globals().get('CADENCE_MIN',5)} min) — leave the kernel running.")


In [ ]:
# === Cell 9 — Stop guardian (fast-cancel with timeout) ===
import asyncio

if 'guard_task' in globals() and guard_task is not None:
    guard_task.cancel()
    try:
        # Wait briefly for a clean exit; don't hang if it's mid HTTP
        await asyncio.wait_for(asyncio.shield(guard_task), timeout=3)
        print("🛑 Guardian stopped cleanly.")
    except asyncio.TimeoutError:
        print("⚠️ Cancel requested; task is still unwinding a blocking call. It will exit shortly.")
    except Exception as e:
        print(f"stop error: {e}")
    finally:
        guard_task = None
else:
    print("No guardian task running.")